# Propensity Score & DML

## Propensity Score Matching - Binary Treatment

In [165]:
import pandas as pd

data = pd.read_csv("../data/matheus_data/learning_mindset.csv")

data.head()

,schoolid,intervention,achievement_score,success_expect,ethnicity,gender,frst_in_family,school_urbanicity,school_mindset,school_achievement,school_ethnic_minority,school_poverty,school_size
0,76,1,0.277359,6,4,2,1,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
1,76,1,-0.449646,4,12,2,1,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
2,76,1,0.769703,6,4,2,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
3,76,1,-0.121763,6,4,2,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
4,76,1,1.526147,6,4,1,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757


### Propensity Score Estimation

In [205]:
from scipy.special import logit
import numpy as np
from causalml.propensity import ElasticNetPropensityModel

categ = ["ethnicity","gender","school_urbanicity"]
cont  = ["school_mindset","school_achievement","school_ethnic_minority","school_poverty","school_size"]
X = pd.get_dummies(data[categ + cont], columns=categ, drop_first=True)

pm = ElasticNetPropensityModel(
    random_state=42,
    max_iter=5000
)
ps = pm.fit_predict(X.values, data["intervention"].values)
logit_ps  = logit(ps)
zlogit_ps = (logit_ps - logit_ps.mean()) / logit_ps.std(ddof=1)

df = data.copy()
df["ps_logit_z"] = zlogit_ps

- **logit(PS) 표준편차**: Propensity Score를 logit 변환 후 표준화하여, 매칭 시 caliper 거리 기준에 활용


### Matching - ATT

In [206]:
from causalml.match import NearestNeighborMatch

df_match = pd.concat([df[["intervention", 'achievement_score', "ps_logit_z"]], X], axis=1)

matcher_att = NearestNeighborMatch(
    caliper=0.2,
    replace=False,
    ratio=1,
    shuffle=False,
    random_state=42,
    treatment_to_control=True
)

matched_att = matcher_att.match(
    data=df_match, treatment_col="intervention", score_cols=["ps_logit_z"]
)

ATT = (
    matched_att.loc[matched_att["intervention"]==1, 'achievement_score'].mean() 
    - matched_att.loc[matched_att["intervention"]==0, 'achievement_score'].mean()
)
print("ATT:", round(ATT, 4))

ATT: 0.4983


- **caliper=0.2**: logit(PS) 표준편차의 20% 이내에서만 매칭
- **replace=False**: 비복원 매칭으로 동일 대조군 중복 사용 방지

### Balance Check - ATT

### Matching - ATC

In [207]:
from causalml.match import NearestNeighborMatch

matcher_atc = NearestNeighborMatch(
    caliper=0.2,
    replace=True,
    ratio=1,
    shuffle=False,
    random_state=42,
    treatment_to_control=False
)

matched_atc = matcher_atc.match(
    data=df_match, treatment_col="intervention", score_cols=["ps_logit_z"]
)

ATC = (
    matched_atc.loc[matched_atc["intervention"]==1, "achievement_score"].mean()
    - matched_atc.loc[matched_atc["intervention"]==0, "achievement_score"].mean() 
)
print("ATC:", round(ATC, 4))

ATC: 0.3843


- **replace=True**: 후보군 부족으로 복원 매칭 허용
  - 매칭 실패 시 caliper 완화, 복원 매칭, k:1 매칭 확대로 대응

### Balance Check - ATC

### ATE

In [208]:
p_t = float(df["intervention"].mean())
ATE = ATT * p_t + ATC * (1 - p_t)
print("ATE:", round(ATE, 4))

ATE: 0.4214


### 질문이나 의견을 남겨주세요.
<script src="https://utteranc.es/client.js"
        repo="CausalInferenceLab/awesome-causal-inference-python"
        issue-term="pathname"
        theme="github-light"
        crossorigin="anonymous"
        async>
</script>